# L04 – Lösningar

In [ ]:
import math
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader, Subset, random_split
from torchvision import datasets, transforms

DATA_DIR = "../data"

### Övning 1

In [ ]:
class SquaresDataset(Dataset):
    def __init__(self, n):
        self.n = n

    def __len__(self):
        return self.n

    def __getitem__(self, idx):
        # Exemplen beräknas när de efterfrågas, och ingenting lagras.
        # Så fungerar också dataset som läser bilder från disk.
        return torch.tensor([float(idx)]), torch.tensor([float(idx ** 2)])

sq = SquaresDataset(100)
assert len(sq) == 100
x, y = sq[7]
assert x.item() == 7.0 and y.item() == 49.0 and x.dtype == torch.float32
xb, yb = next(iter(DataLoader(sq, batch_size=10)))
assert xb.shape == (10, 1)
print("Rätt!")

### Övning 2

In [ ]:
raw_train = datasets.FashionMNIST(DATA_DIR, train=True, download=True, transform=transforms.ToTensor())
train_ds, val_ds = random_split(raw_train, [50_000, 10_000], generator=torch.Generator().manual_seed(42))

small_loader = DataLoader(Subset(train_ds, range(5000)), batch_size=64, shuffle=True)
assert len(small_loader.dataset) == 5000
assert len(small_loader) == math.ceil(5000 / 64)
print("Rätt!", len(small_loader), "batcher")   # 79: 78 hela batcher och en med 8 exempel

### Övning 3
**Felet:** medelvärde och standardavvikelse beräknas på *alla* 60 000 bilder, alltså även de 10 000 som blir valideringsdata. Information från valideringsmängden läcker då in i förbehandlingen.

**Spelar det roll?** Här är effekten liten, eftersom bilderna är lika varandra. Principen är ändå viktig. Med tabelldata, outliers eller tidsserier kan läckage ge kraftigt överskattade resultat. Det är exakt samma fel som att köra `StandardScaler().fit(X)` på all data före `train_test_split`.

**Rätt ordning:** dela först och beräkna statistiken sedan.

In [ ]:
generator = torch.Generator().manual_seed(42)
train_part, val_part = random_split(raw_train, [50_000, 10_000], generator=generator)
train_pixels = raw_train.data[train_part.indices].float() / 255
MEAN, STD = train_pixels.mean().item(), train_pixels.std().item()
print(f"MEAN={MEAN:.4f} STD={STD:.4f}")